# Generate Last-Layer Attention Matrices Across Inputs

This Generate notebook computes and caches the data consumed by `Visualize-last-layer-attention-matrices-across-inputs.ipynb`. The companion Visualize notebook presents head-averaged last-layer attention matrices for representative sequences from the main numerical input families at $N = 500$ using Llama-3.2-3B, as presented under **Attention-Induced Token Graphs: Construction and Visualization Robustness** in the appendix.

Import dependencies; define the model, 500-token analysis window, relative cache schema, and random seeds.

In [ ]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import gc
import random
import re
from pathlib import Path

import numpy as np
import torch
from scipy.integrate import solve_ivp
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Model, 500-token analysis window, and relative cache path
MODEL_NAME = 'meta-llama/Llama-3.2-3B'
REFERENCE_CONTEXT_TOKENS = 1000
ANALYSIS_CONTEXT_TOKENS = 500
CONTEXT_FRAC = ANALYSIS_CONTEXT_TOKENS / REFERENCE_CONTEXT_TOKENS
CONTEXT_DISPLAY_LABEL = f'{ANALYSIS_CONTEXT_TOKENS} tokens'
ATTENTION_DATA_DIR = Path('processed_last_layer_attention_data')
LAST_LAYER_ATTENTION_SCHEMA_VERSION = 'last_layer_attention_matrix_v1'

# Remove an optional leading BOS token and renormalize retained attention rows
ATTENTION_DROP_BOS = True
ATTENTION_RENORMALIZE_AFTER_DROP = True

# Seed sequence generation and model inference
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# Input-family order, display labels, and cache schema
DATA_ORDER = [
    'Constant 500',
    'Logistic r=3.2',
    'Logistic r=3.5',
    'Logistic r=3.55',
    'Logistic r=3.566',
    'Logistic r=3.5692',
    'Logistic r=3.5698',
    'Logistic r=3.9',
    'Logistic r=4.0',
    'Lorenz',
]

DATA_TITLE_SUFFIXES = {
    'Logistic r=3.2': 'period 2',
    'Logistic r=3.5': 'period 4',
    'Logistic r=3.55': 'period 8',
    'Logistic r=3.566': 'period 16',
    'Logistic r=3.5692': 'period 32',
    'Logistic r=3.5698': 'period 64',
    'Logistic r=3.9': 'chaotic',
    'Logistic r=4.0': 'chaotic',
}

ATTENTION_DATA_ARRAY_KEYS = (
    'schema_version',
    'model_name',
    'data_name',
    'reference_context_tokens',
    'analysis_context_tokens',
    'context_frac',
    'ctx_len',
    'drop_bos',
    'renormalize_after_drop',
    'layer_index',
    'layer_index_zero_based',
    'num_model_layers',
    'num_heads',
    'token_start',
    'token_ids',
    'token_texts',
    'is_delimiter',
    'input_values',
    'attention_heads',
    'attention_mean',
)


def panel_title(data_name):
    suffix = DATA_TITLE_SUFFIXES.get(str(data_name))
    return f'{data_name} ({suffix})' if suffix else str(data_name)


def context_display_label(num_tokens=ANALYSIS_CONTEXT_TOKENS):
    return f'{int(num_tokens)} tokens'


def context_fraction_label(frac):
    pct = int(round(100 * float(frac)))
    return f'context{pct}'


def safe_data_stem(data_name, frac):
    name = str(data_name).replace('=', '_').replace('.', '_')
    name = re.sub(r'[^A-Za-z0-9]+', '_', name).strip('_').lower()
    return f'{name}_{context_fraction_label(frac)}'


def attention_data_path(data_name, frac=CONTEXT_FRAC, data_dir=ATTENTION_DATA_DIR):
    return Path(data_dir) / f'{safe_data_stem(data_name, frac)}_last_layer_attention.npz'


def resolve_last_layer_indices(num_layers):
    num_layers = int(num_layers)
    if num_layers <= 0:
        raise ValueError(f'Invalid num_layers={num_layers}.')
    layer_index = num_layers
    layer_index_zero_based = num_layers - 1
    return int(layer_index), int(layer_index_zero_based)


Define representative constant, periodic and chaotic logistic-map, and Lorenz sequences.

In [ ]:
def lorenz(t, y, sigma, rho, beta):
    return [
        sigma * (y[1] - y[0]),
        y[0] * (rho - y[2]) - y[1],
        y[0] * y[1] - beta * y[2],
    ]


# Sample and quantize a representative Lorenz x-coordinate trajectory
sigma = 10.0
rho = 28.0
beta = 8.0 / 3.0
initial_conditions = [-8, 7, 27]
t_span = (0, 50)
t_eval = np.linspace(t_span[0], t_span[1], 500)

sol = solve_ivp(
    lorenz,
    t_span,
    initial_conditions,
    args=(sigma, rho, beta),
    t_eval=t_eval,
)

x_vals = sol.y[0]
x_min = x_vals.min()
x_max = x_vals.max()
x_integers = np.round(50 + (x_vals - x_min) / (x_max - x_min) * 900).astype(int)
N = len(x_integers)


In [ ]:
# Periodic and chaotic logistic-map regimes
periodic_logistic_periods = {
    3.2: 2,
    3.5: 4,
    3.55: 8,
    3.566: 16,
    3.5692: 32,
    3.5698: 64,
}
logistic_rs = list(periodic_logistic_periods.keys()) + [3.9, 4.0]
logistic_expected_periods = {r: periodic_logistic_periods.get(r) for r in logistic_rs}
x0 = 0.2
logistic_long_n = 20000
logistic_settle_tol = 1e-10


def logistic_sequence(r, x0, n):
    x = np.empty(n, dtype=float)
    x[0] = x0
    for i in range(1, n):
        x[i] = r * x[i - 1] * (1 - x[i - 1])
    return x


def settle_time_float(x, p, tol=1e-10):
    x = np.asarray(x, dtype=float)
    if p is None or p <= 0 or p >= x.size:
        return None
    err = np.abs(x[p:] - x[:-p])
    suffix_max = np.maximum.accumulate(err[::-1])[::-1]
    hits = np.flatnonzero(suffix_max < tol)
    return int(hits[0]) if hits.size else None


def quantize_logistic_window(vals):
    vals = np.asarray(vals, dtype=float)
    vmin, vmax = vals.min(), vals.max()
    if vmax == vmin:
        return np.full(vals.shape[0], 500, dtype=int)
    return np.round(50 + (vals - vmin) / (vmax - vmin) * 900).astype(int)


def build_burned_in_logistic_window(r, x0, n_window, expected_period=None, n_long=20000, tol=1e-10):
    if expected_period is None:
        return logistic_sequence(r, x0, n_window), 0

    x_long = logistic_sequence(r, x0, n_long)
    t_settle = settle_time_float(x_long, expected_period, tol=tol)
    if t_settle is None:
        raise ValueError(f'Could not find settle time for r={r} with expected period {expected_period}.')
    if t_settle + n_window > x_long.size:
        raise ValueError(f'Need a longer trajectory for r={r}; settle index {t_settle} leaves fewer than {n_window} states.')
    return np.asarray(x_long[t_settle:t_settle + n_window], dtype=float), int(t_settle)


# Use settled periodic windows and untrimmed chaotic windows
logistic_sequences = {}
logistic_integers = {}
logistic_window_start_indices = {}
for r in logistic_rs:
    vals, start_idx = build_burned_in_logistic_window(
        r,
        x0=x0,
        n_window=N,
        expected_period=logistic_expected_periods[r],
        n_long=logistic_long_n,
        tol=logistic_settle_tol,
    )
    logistic_sequences[r] = vals
    logistic_integers[r] = quantize_logistic_window(vals)
    logistic_window_start_indices[r] = int(start_idx)

# Assemble one representative 500-state source sequence per input family
input_data_types = {
    'Constant 500': np.full(N, 500, dtype=int),
    'Logistic r=3.2': logistic_integers[3.2],
    'Logistic r=3.5': logistic_integers[3.5],
    'Logistic r=3.55': logistic_integers[3.55],
    'Logistic r=3.566': logistic_integers[3.566],
    'Logistic r=3.5692': logistic_integers[3.5692],
    'Logistic r=3.5698': logistic_integers[3.5698],
    'Logistic r=3.9': logistic_integers[3.9],
    'Logistic r=4.0': logistic_integers[4.0],
    'Lorenz': x_integers,
}

assert list(input_data_types.keys()) == DATA_ORDER


Initialize the tokenizer and load Llama-3.2-3B with eager attention.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model_attn = None
num_layers = None

# Check the comma delimiter to occupy exactly one token
comma_token_ids = tokenizer(',', add_special_tokens=False).input_ids
if len(comma_token_ids) != 1:
    raise ValueError(f"Expected one token for ',', got {comma_token_ids}")
comma_token_id = int(comma_token_ids[0])


def load_attention_model(model_name=MODEL_NAME):
    global model_attn, num_layers
    if model_attn is None:
        model_attn = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map='auto',
            torch_dtype=torch.float16,
            attn_implementation='eager',
        )
        model_attn.eval()
        num_layers = int(model_attn.config.num_hidden_layers)
    return model_attn


Build comma-delimited 500-token prompts and extract final-layer attention matrices.

In [ ]:
def get_token_start_index(input_ids, bos_token_id=None, drop_bos=True):
    if not drop_bos:
        return 0
    if input_ids is None or bos_token_id is None:
        return 0

    if isinstance(input_ids, torch.Tensor):
        first_id = int(input_ids.flatten()[0].item())
    else:
        first_id = int(np.asarray(input_ids).flatten()[0])

    return 1 if first_id == int(bos_token_id) else 0


def _build_input_sequence_from_frac(sequence, frac, total_n):
    frac_use = float(frac)
    if frac_use >= 1.0:
        ctx_len = int(total_n)
    else:
        ctx_len = max(1, int(round(frac_use * (total_n - 1))))
    ctx_len = min(int(total_n), ctx_len)
    context_window = np.asarray(sequence[:ctx_len], dtype=int)
    # Format each numerical state as three digits followed by a comma
    input_sequence = ','.join([f'{x:03d}' for x in context_window]) + ','
    return input_sequence, ctx_len, context_window


def _extract_last_layer_attention(
    outputs_attn,
    input_ids,
    bos_token_id=None,
    drop_bos=ATTENTION_DROP_BOS,
    renormalize_after_drop=ATTENTION_RENORMALIZE_AFTER_DROP,
):
    if outputs_attn.attentions is None:
        raise ValueError('Model did not return attentions. Use eager attention implementation.')

    # Resolve the final attention layer and align it with post-BOS tokens
    num_attn_layers = len(outputs_attn.attentions)
    layer_index, layer_index_zero_based = resolve_last_layer_indices(num_attn_layers)
    layer_attn = outputs_attn.attentions[layer_index_zero_based][0].float()
    token_start = get_token_start_index(input_ids, bos_token_id=bos_token_id, drop_bos=drop_bos)

    # Remove an optional leading BOS from both query and key axes
    A = layer_attn[:, token_start:, token_start:]
    if renormalize_after_drop and token_start > 0:
        row_sums = A.sum(dim=-1, keepdim=True)
        row_sums = torch.where(row_sums == 0, torch.ones_like(row_sums), row_sums)
        A = A / row_sums

    attention_heads = A.detach().cpu().numpy().astype(np.float32)
    # Mass and uniform weighting are equivalent for the models studied here; mass also supports sparse-attention variants
    attention_mean = attention_heads.mean(axis=0, dtype=np.float64).astype(np.float32)
    return attention_heads, attention_mean, int(token_start), int(layer_index), int(layer_index_zero_based)


def compute_last_layer_attention_data(data_name, frac=CONTEXT_FRAC):
    if data_name not in input_data_types:
        raise KeyError(f'Unknown data_name={data_name!r}. Available: {list(input_data_types)}')

    sequence = input_data_types[data_name]
    input_sequence, ctx_len, context_window = _build_input_sequence_from_frac(sequence, frac, N)

    attn_model = load_attention_model(MODEL_NAME)
    attn_device = next(attn_model.parameters()).device
    num_model_layers = int(attn_model.config.num_hidden_layers)

    inputs = tokenizer(input_sequence, return_tensors='pt').to(attn_device)
    input_ids = inputs.input_ids[0]
    input_ids_np = input_ids.detach().cpu().numpy()

    # Request full eager-attention tensors without gradients
    with torch.no_grad():
        outputs = attn_model(
            **inputs,
            output_attentions=True,
            return_dict=True,
        )

    attention_heads, attention_mean, token_start, layer_index, layer_index_zero_based = _extract_last_layer_attention(
        outputs,
        input_ids=input_ids,
        bos_token_id=tokenizer.bos_token_id,
        drop_bos=ATTENTION_DROP_BOS,
        renormalize_after_drop=ATTENTION_RENORMALIZE_AFTER_DROP,
    )

    token_ids = np.asarray(input_ids_np[token_start:], dtype=np.int64)
    token_texts = np.asarray([tokenizer.decode([int(token_id)]) for token_id in token_ids])
    is_delimiter = token_ids == comma_token_id

    # Validate layer selection, token alignment, and row normalization
    if int(layer_index) != int(num_model_layers):
        raise ValueError(f'Expected final layer {num_model_layers}, got {layer_index}.')
    if attention_heads.ndim != 3:
        raise ValueError(f'attention_heads must be 3D, got shape {attention_heads.shape}.')
    if attention_heads.shape[1:] != attention_mean.shape:
        raise ValueError(f'Head/mean shape mismatch: {attention_heads.shape} vs {attention_mean.shape}.')
    if attention_mean.shape != (token_ids.size, token_ids.size):
        raise ValueError(f'Attention/token shape mismatch: {attention_mean.shape} vs {token_ids.size} tokens.')
    if ATTENTION_RENORMALIZE_AFTER_DROP:
        max_row_error = float(np.max(np.abs(attention_heads.sum(axis=-1) - 1.0)))
        if max_row_error > 2e-3:
            raise ValueError(f'Attention rows are not normalized after BOS drop; max row error={max_row_error:.4g}.')

    # Store the arrays and metadata required by the companion visualization
    processed_data = {
        'schema_version': np.asarray(LAST_LAYER_ATTENTION_SCHEMA_VERSION),
        'model_name': np.asarray(MODEL_NAME),
        'data_name': np.asarray(str(data_name)),
        'reference_context_tokens': np.asarray(REFERENCE_CONTEXT_TOKENS, dtype=np.int32),
        'analysis_context_tokens': np.asarray(ANALYSIS_CONTEXT_TOKENS, dtype=np.int32),
        'context_frac': np.asarray(float(frac), dtype=np.float64),
        'ctx_len': np.asarray(ctx_len, dtype=np.int32),
        'drop_bos': np.asarray(bool(ATTENTION_DROP_BOS)),
        'renormalize_after_drop': np.asarray(bool(ATTENTION_RENORMALIZE_AFTER_DROP)),
        'layer_index': np.asarray(layer_index, dtype=np.int32),
        'layer_index_zero_based': np.asarray(layer_index_zero_based, dtype=np.int32),
        'num_model_layers': np.asarray(num_model_layers, dtype=np.int32),
        'num_heads': np.asarray(attention_heads.shape[0], dtype=np.int32),
        'token_start': np.asarray(token_start, dtype=np.int32),
        'token_ids': token_ids,
        'token_texts': token_texts,
        'is_delimiter': np.asarray(is_delimiter, dtype=bool),
        'input_values': np.asarray(context_window, dtype=np.int32),
        'attention_heads': np.asarray(attention_heads, dtype=np.float32),
        'attention_mean': np.asarray(attention_mean, dtype=np.float32),
    }

    # Release large model outputs between input families
    del outputs, inputs, input_ids
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return processed_data


Save archives and validate schema, metadata, shapes, and row normalization.

In [ ]:
def _atomic_savez_compressed(path, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(path.name + '.tmp')
    with tmp_path.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    tmp_path.replace(path)


def save_last_layer_attention_data(data_name, frac, processed_data, data_dir=ATTENTION_DATA_DIR):
    path = attention_data_path(data_name, frac=frac, data_dir=data_dir)

    missing = [key for key in ATTENTION_DATA_ARRAY_KEYS if key not in processed_data]
    if missing:
        raise KeyError(f'Missing required attention arrays: {missing}')

    arrays_to_save = {key: processed_data[key] for key in ATTENTION_DATA_ARRAY_KEYS}
    _atomic_savez_compressed(path, **arrays_to_save)
    return path


def load_last_layer_attention_data(data_name, frac=CONTEXT_FRAC, data_dir=ATTENTION_DATA_DIR):
    path = attention_data_path(data_name, frac=frac, data_dir=data_dir)
    if not path.exists():
        raise FileNotFoundError(path)

    with np.load(path, allow_pickle=False) as npz:
        processed_data = {key: npz[key] for key in npz.files}

    missing = [key for key in ATTENTION_DATA_ARRAY_KEYS if key not in processed_data]
    if missing:
        raise KeyError(f'{path} is missing required arrays: {missing}')
    extra = sorted(set(processed_data) - set(ATTENTION_DATA_ARRAY_KEYS))
    if extra:
        raise KeyError(f'{path} contains unexpected arrays: {extra}')
    return processed_data


def _npz_scalar_to_str(data, key):
    return str(np.asarray(data[key]).item())


def last_layer_attention_data_complete(data_name, frac=CONTEXT_FRAC, data_dir=ATTENTION_DATA_DIR):
    try:
        data = load_last_layer_attention_data(data_name, frac=frac, data_dir=data_dir)
    except (FileNotFoundError, OSError, KeyError, ValueError):
        return False

    if _npz_scalar_to_str(data, 'schema_version') != LAST_LAYER_ATTENTION_SCHEMA_VERSION:
        return False
    if _npz_scalar_to_str(data, 'model_name') != MODEL_NAME:
        return False
    if _npz_scalar_to_str(data, 'data_name') != str(data_name):
        return False
    if not np.isclose(float(np.asarray(data['context_frac']).item()), float(frac)):
        return False
    if bool(np.asarray(data['drop_bos']).item()) != bool(ATTENTION_DROP_BOS):
        return False
    if bool(np.asarray(data['renormalize_after_drop']).item()) != bool(ATTENTION_RENORMALIZE_AFTER_DROP):
        return False

    token_ids = np.asarray(data['token_ids'], dtype=np.int64)
    is_delimiter = np.asarray(data['is_delimiter'], dtype=bool)
    attention_heads = np.asarray(data['attention_heads'], dtype=np.float32)
    attention_mean = np.asarray(data['attention_mean'], dtype=np.float32)
    n_tokens = int(token_ids.size)
    n_heads = int(np.asarray(data['num_heads']).item())

    shape_ok = (
        is_delimiter.shape == (n_tokens,)
        and attention_heads.shape == (n_heads, n_tokens, n_tokens)
        and attention_mean.shape == (n_tokens, n_tokens)
    )
    if not shape_ok:
        return False
    if ATTENTION_RENORMALIZE_AFTER_DROP and not np.allclose(attention_heads.sum(axis=-1), 1.0, atol=2e-3):
        return False
    return True


Generate in input-family order.

In [ ]:
generated_paths = []
skipped_paths = []

# Reuse compatible archives; rebuild stale or incomplete files
for data_name in tqdm(DATA_ORDER, desc='Last-layer attention data', unit='input'):
    output_path = attention_data_path(data_name, frac=CONTEXT_FRAC)
    if last_layer_attention_data_complete(data_name, frac=CONTEXT_FRAC):
        skipped_paths.append(output_path)
        continue

    if output_path.exists():
        print(f'Rebuilding incomplete or outdated archive: {output_path}')

    processed = compute_last_layer_attention_data(data_name, frac=CONTEXT_FRAC)
    saved_path = save_last_layer_attention_data(data_name, CONTEXT_FRAC, processed)
    generated_paths.append(saved_path)

print(f'Generated {len(generated_paths)} archive(s); skipped {len(skipped_paths)} existing archive(s).')
print(f'Last-layer attention data directory: {ATTENTION_DATA_DIR}')
